## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, lower, upper, count, regexp_replace

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.product_category_name_translation")
df.display()

## Overview about the table

In [0]:
# Table info
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

## Transformations

### 1. Trim all whitespaces from string columns

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))


### 2. Normalize improperly represented nulls in string columns

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-"]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name))
        )

### 3. Standardize string columns

In [0]:
df = df.withColumn(
    "product_category_name_english",
    lower(regexp_replace(col("product_category_name_english"), "_", " "))
).withColumn(
         "product_category_name",
         lower(regexp_replace(col("product_category_name"), "_", " "))
)

### 4. Remove duplicate rows

In [0]:
df = df.dropDuplicates()

## Quality Checks

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Distinct categories: {df.select('product_category_name').distinct().count()}")
df.display()

## Write it to silver layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.product_category_translation")